# Train a detector on a Hugging Face QA dataset

The four published detectors each belong to one model. Scoring any other model means
training a detector for it, and this notebook does that end to end on
[TriviaQA](https://huggingface.co/datasets/mandarjoshi/trivia_qa) — the set the paper
trains on — in five steps:

1. **Draw questions** from TriviaQA, which ships a gold short answer and its accepted aliases.
2. **Answer them** with the model you want to score, keeping `top_logprobs` for every token.
3. **Label** each answer against the gold aliases: `1` marks a hallucination.
4. **Fit** WEPR on the labelled responses and score it on a held-out quarter.
5. **Save** the weights, and reload them the way any published detector is loaded.

Nothing here is specific to TriviaQA. Any short-form QA set on the Hub works — only the
column names in step 1 change — and the same is true of the model: the requirement is an
endpoint that returns `top_logprobs`, not a particular provider.

## Prerequisites

```bash
uv pip install "artefactual[adapters]" datasets
```

| Variable | Required | What it is |
|---|---|---|
| `OPENAI_BASE_URL` | yes | Any OpenAI-compatible endpoint that returns `top_logprobs` |
| `OPENAI_API_KEY` | yes | Its key |
| `OPENAI_MODEL` | no | The model being scored; the detector you train belongs to it |

**A detector belongs to the model it was trained on.** The weights read that model's
confidence, so scoring a different one with them is not supported — retrain instead. This
notebook is not executed when the documentation is built, because it generates against a
live endpoint; the numbers you see are the ones your run produces.

## 1. Draw the questions

`rc.nocontext` is the closed-book configuration: question and answer, no evidence
documents. Each row carries `answer.value` — the gold short answer — and `answer.aliases`,
the other spellings that count as correct.

The sample is shuffled before it is sliced, because the split arrives grouped by source;
the head of it is a narrower sample than the same count drawn at random. 300 questions
takes a few minutes to generate and is enough to fit on. If the interval in step 5 is too
wide to read, raise it.

In [ ]:
import os

MODEL = os.environ.get("OPENAI_MODEL", "mistralai/Ministral-8B-Instruct-2410")

# Ranks kept per token. It is part of the feature definition, not a batch size: WEPR fits
# one coefficient per rank, so the value used here is the value the detector must later be
# loaded and scored at. Every published detector uses 15.
K = 15

N_QUESTIONS = 300
SEED = 42

In [ ]:
from datasets import load_dataset

trivia_qa = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
sample = trivia_qa.shuffle(seed=SEED).select(range(N_QUESTIONS))

questions = [
    {
        "question": row["question"],
        "short_answer": row["answer"]["value"],
        # `value` is not always in `aliases`, so it is added rather than assumed present.
        "answers": [row["answer"]["value"], *row["answer"]["aliases"]],
    }
    for row in sample
]

print(f"{len(questions)} questions")
print(questions[0]["question"], "->", questions[0]["short_answer"])

## 2. Answer them, keeping the log-probabilities

The detector reads the token distribution behind an answer, so `logprobs=True` and
`top_logprobs=K` are what make a response scoreable at all. A response generated without
them carries nothing to score, and one generated with fewer than `K` ranks is refused when
it reaches the parser rather than silently zero-filled.

The prompt and the sampling settings are the paper's (§4.1.2): non-greedy decoding at
`T = 1.0`, `top_p = 1.0`. Non-greedy is the point — the method measures hesitation in the
raw distribution.

Requests are threaded because 300 sequential round trips is the slow part of this
notebook, not the fitting. Failures return `None` and are dropped as a pair with their
question, so a rate-limited request costs one example rather than the run.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI

client = OpenAI()  # reads OPENAI_BASE_URL and OPENAI_API_KEY

PROMPT = """You are a useful assistant that help finding short and precise answers for a given query or question.
            Please keep your output AS SHORT AND CONCISE AS POSSIBLE.
            Here is the query :
            {query}
            """


def answer(question: dict):
    try:
        return client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": PROMPT.format(query=question["question"])}],
            logprobs=True,
            top_logprobs=K,
            temperature=1.0,
            top_p=1.0,
            max_completion_tokens=200,
        )
    except Exception as error:  # noqa: BLE001 - one failed request should not cost the run
        print(f"dropped {question['question'][:60]!r}: {error}")
        return None


with ThreadPoolExecutor(max_workers=8) as pool:
    generated = list(pool.map(answer, questions))

# `map` preserves order, so a question and its response stay paired through the filter.
pairs = [(q, r) for q, r in zip(questions, generated) if r is not None]
questions, responses = [q for q, _ in pairs], [r for _, r in pairs]
print(f"{len(responses)}/{len(generated)} generated")

In [ ]:
# Confirm the ranks actually came back: an endpoint that ignores `top_logprobs` returns a
# valid completion carrying nothing to score, and this is the cheapest place to notice.
width = len(responses[0].choices[0].logprobs.content[0].top_logprobs)
assert width >= K, f"endpoint returned {width} ranks per token, need {K}"

print(responses[0].choices[0].message.content)

## 3. Label the answers

`1` marks a hallucination. TriviaQA's aliases are what makes this a two-line step rather
than a second LLM pass: an answer is correct when it contains one of them, after both
sides are normalised the way the dataset's own metric normalises — lowercased,
punctuation and articles dropped.

Comparison is on token runs rather than raw substrings, so the alias `US` does not match
inside *because*.

**This is a proxy for the paper's labels.** The ECIR pipeline grades with an LLM judge,
which reads a hedged or reworded answer that alias matching scores as wrong. Alias
matching costs nothing and needs no second model, which is the trade a quick start wants;
the judge, and the whole batch pipeline around it, is in
[`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir).

In [ ]:
import string

import numpy as np

ARTICLES = {"a", "an", "the"}


def normalize(text: str) -> list[str]:
    stripped = text.lower().translate(str.maketrans("", "", string.punctuation))
    return [word for word in stripped.split() if word not in ARTICLES]


def contains(haystack: list[str], needle: list[str]) -> bool:
    return bool(needle) and any(haystack[i : i + len(needle)] == needle for i in range(len(haystack) - len(needle) + 1))


def is_hallucination(response, question: dict) -> int:
    generated = normalize(response.choices[0].message.content or "")
    return 0 if any(contains(generated, normalize(a)) for a in question["answers"]) else 1


y = np.array(list(map(is_hallucination, responses, questions)))
print(f"{y.sum()}/{len(y)} labelled as hallucinations ({y.mean():.0%})")

In [ ]:
# Both classes are needed, and this is where a run fails cheaply rather than inside `fit`.
# All-correct means the questions are too easy for this model; all-wrong usually means the
# model is not answering in the expected short form.
assert 0 < y.sum() < len(y), f"labels are single-class ({y.sum()}/{len(y)}); the questions need to be harder or easier"

for label in (0, 1):
    index = int(np.argmax(y == label))
    print(f"[{'hallucination' if label else 'grounded'}] {questions[index]['question']}")
    print(f"    gold: {questions[index]['short_answer']}")
    print(f"    said: {responses[index].choices[0].message.content.strip()[:120]}\n")

## 4. Fit the detector

`trainable=True` returns an unfitted pipeline — parser, entropy reduction, logistic
regression — that takes the raw responses. There is no feature extraction step to write:
the pipeline reads the completions as they came back from the endpoint.

The split is stratified so the hallucination rate is the same on both sides, and it
happens before the fit so the numbers in step 5 describe responses the detector has never
seen.

In [ ]:
from sklearn.model_selection import train_test_split

from artefactual.scoring import wepr

x_train, x_test, y_train, y_test = train_test_split(responses, y, test_size=0.25, stratify=y, random_state=SEED)

detector = wepr(k=K, trainable=True).fit(x_train, y_train)
print(f"fitted on {len(y_train)} responses, holding out {len(y_test)}")
detector

## 5. Score it on the held-out quarter

Two different questions, and both are worth reading. **ROC-AUC** scores the *ranking* —
whether hallucinations get higher scores than grounded answers — which is what governs
triage by score and what the paper reports. The **classification report** scores the
decisions at the 0.5 threshold, and recall on the `hallucination` row is the fraction of
hallucinations actually flagged.

A detector can rank well and still decide poorly at 0.5. Only the AUC carries over to a
different threshold, so pick the threshold from these scores rather than assuming 0.5;
[the scoring guide](https://artefactory.github.io/artefactual/guide/scoring.html) covers
how.

For reference, the paper reports 85.8 ROC-AUC for WEPR on TriviaQA with
`Ministral-8B-2410`, fitted on far more than 300 questions and labelled by an LLM judge.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

scores = detector.predict_proba(x_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, scores):.3f}")
print(classification_report(y_test, scores >= 0.5, target_names=["grounded", "hallucination"], zero_division=0))

## 6. Save the weights, and use them

The file is the same `.skops` format the published detectors ship in, so it loads through
the same call — a repository id, a path, either one. `k` has to be the value the weights
were fitted at, and passing another one raises rather than mis-shaping the score.

From here it is an ordinary detector: `predict_proba` for a score per response,
`predict_token_proba` for where in the answer the model started drifting.

In [ ]:
path = detector.save_estimator("wepr-triviaqa.skops")

reloaded = wepr(path, k=K)
print(f"P(hallucination) = {reloaded.predict_proba(responses[0])[0, 1]:.3f}")
print(responses[0].choices[0].message.content.strip())

## Where to go next

- **More data.** 300 questions is enough to see a signal, not enough to pin it down. The
  fit is seconds; the generation is the cost, so raise `N_QUESTIONS` and rerun.
- **A better label.** Alias matching calls a correct-but-reworded answer a hallucination.
  The LLM judge in [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir)
  is the paper's procedure, and that README also runs generation as a `vllm run-batch`
  job, which is the practical way to do this at thousands of questions.
- **Another dataset.** Only step 1 changes. Training data should resemble the traffic
  being scored: the paper's numbers drop 10–20 points when a TriviaQA-trained detector
  meets WebQuestions.
- **`epr` instead of `wepr`.** Same call, same data, one feature instead of `2k`. WEPR
  beat EPR on every row of the paper's table, which is why it is the default here.